# Doing some EDA to understand ArXiv problem set

Using our dataset loading tools to do some basic Q&A on ArXiv dataset make up. Mostly for understanding geocoding feasibility.

*Aniket Pant*

In [17]:
import numpy as np
import json
import pandas as pd

In [18]:
from arxiv_map.warehouse import ArxivMetadataWarehouse, CometWarehouse
from arxiv_map.merge import CometArxivMerger


arxiv = ArxivMetadataWarehouse()
comet = CometWarehouse()

# by default, this will not build if warehouse already exists at default path
arxiv.build()
comet.build()

In [19]:
%time
comet_rows = comet.find_by_ids(["arXiv:1109.3791", "1109.3792v1"])
arxiv_rows = arxiv.find_by_ids([row["arxiv_id"] for row in comet_rows])

CPU times: user 4 μs, sys: 3 μs, total: 7 μs
Wall time: 12.9 μs


In [5]:
arxiv_rows

{'1109.3791': {'id': '1109.3791',
  'title': 'WebCloud: Recruiting web browsers for content distribution',
  'abstract': "We are at the beginning of a shift in how content is created and exchanged\nover the web. While content was previously created primarily by a small set of\nentities, today, individual users -- empowered by devices like digital cameras\nand services like online social networks -- are creating content that\nrepresents a significant fraction of Internet traffic. As a result, content\ntoday is increasingly generated and exchanged at the edge of the network.\nUnfortunately, the existing techniques and infrastructure that are still used\nto serve this content, such as centralized content distribution networks, are\nill-suited for these new patterns of content exchange. In this paper, we take a\nfirst step towards addressing this situation by introducing WebCloud, a content\ndistribution system for online social networking sites that works by re-\npurposing web browsers to

In [6]:
top_categories = arxiv.query("""
    SELECT category, COUNT(DISTINCT arxiv_id) AS paper_count
    FROM arxiv_paper_categories
    WHERE published_at >= DATE '2020-01-01'
    GROUP BY category
    ORDER BY paper_count DESC
    LIMIT 20
""")

In [7]:
top_categories

,category,paper_count
0,cs.LG,116487
1,cs.CV,78530
2,cs.AI,58308
3,quant-ph,43423
4,cs.CL,38923
5,hep-ph,29885
6,hep-th,29301
7,stat.ML,28621
8,cond-mat.mtrl-sci,27748
9,gr-qc,24955


In [11]:
comet_rows

[{'arxiv_id': '1109.3791',
  'doi': '10.48550/arxiv.1109.3791',
  'version': 'v1',
  'prediction': [{'name': 'Fangfei Zhou',
    'affiliations': [{'affiliation': 'Northeastern University',
      'ror_id': None}]},
   {'name': 'Richard Revis',
    'affiliations': [{'affiliation': 'Jandrell, Pearson & Revis Ltd.',
      'ror_id': None}]},
   {'name': 'Liang Zhang',
    'affiliations': [{'affiliation': 'Northeastern University',
      'ror_id': None}]},
   {'name': 'Alan Mislove',
    'affiliations': [{'affiliation': 'Northeastern University',
      'ror_id': None}]},
   {'name': 'Eric Franco',
    'affiliations': [{'affiliation': 'Northeastern University',
      'ror_id': None}]},
   {'name': 'Ravi Sundaram',
    'affiliations': [{'affiliation': 'Northeastern University',
      'ror_id': None}]}]},
 {'arxiv_id': '1109.3792',
  'doi': '10.48550/arxiv.1109.3792',
  'version': 'v1',
  'prediction': [{'name': 'L. Viennot',
    'affiliations': [{'affiliation': 'Laboratoire de didactique André

In [21]:
top20_category_affiliations = comet.query("""
ATTACH 'data/processed/arxiv.duckdb' AS arxiv_db;

WITH top_categories AS (
  SELECT
    category,
    COUNT(DISTINCT arxiv_id) AS paper_count
  FROM arxiv_db.arxiv_paper_categories
  WHERE published_at >= DATE '2020-01-01'
  GROUP BY category
  ORDER BY paper_count DESC
  LIMIT 20
),

eligible_papers AS (
  SELECT DISTINCT
    pc.arxiv_id,
    pc.category
  FROM arxiv_db.arxiv_paper_categories pc
  JOIN top_categories tc
    ON tc.category = pc.category
  WHERE pc.published_at >= DATE '2020-01-01'
),

affiliations AS (
  SELECT
    ep.category,
    regexp_replace(
      lower(trim(affiliation_rows.affiliation.affiliation)),
      '\\s+',
      ' ',
      'g'
    ) AS normalized_affiliation,
    affiliation_rows.affiliation.ror_id AS ror_id
  FROM eligible_papers ep
  JOIN comet_papers cp
    ON cp.arxiv_id = ep.arxiv_id,
    unnest(from_json(cp.prediction_json, '[
      {
        "name": "VARCHAR",
        "affiliations": [
          {
            "affiliation": "VARCHAR",
            "ror_id": "VARCHAR"
          }
        ]
      }
    ]')) AS author_rows(author),
    unnest(author_rows.author.affiliations) AS affiliation_rows(affiliation)
  WHERE affiliation_rows.affiliation.affiliation IS NOT NULL
    AND trim(affiliation_rows.affiliation.affiliation) <> ''
)

SELECT
  category,
  COUNT(*) AS affiliation_mentions,
  COUNT(DISTINCT normalized_affiliation) AS unique_affiliations,
  COUNT(DISTINCT normalized_affiliation) FILTER (WHERE ror_id IS NOT NULL) AS unique_affiliations_with_ror,
  COUNT(DISTINCT normalized_affiliation) FILTER (WHERE ror_id IS NULL) AS unique_affiliations_without_ror
FROM affiliations
GROUP BY category
ORDER BY unique_affiliations DESC;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
top20_category_affiliations

,category,affiliation_mentions,unique_affiliations,unique_affiliations_with_ror,unique_affiliations_without_ror
0,cs.LG,543904,128569,88737,39832
1,cs.CV,433037,85463,58334,27129
2,cs.AI,293040,69582,46939,22643
3,cond-mat.mtrl-sci,214177,55545,45614,9931
4,quant-ph,225933,52277,39726,12552
5,astro-ph.GA,210726,45003,35006,9997
6,cond-mat.mes-hall,151957,35385,28889,6497
7,stat.ML,111651,34565,24432,10133
8,astro-ph.HE,142035,34171,26651,7520
9,cs.CL,202033,33088,20884,12204


In [24]:
top20_total_unique_affiliations = comet.query("""
ATTACH 'data/processed/arxiv.duckdb' AS arxiv_db;

WITH top_categories AS (
  SELECT category
  FROM arxiv_db.arxiv_paper_categories
  WHERE published_at >= DATE '2020-01-01'
  GROUP BY category
  ORDER BY COUNT(DISTINCT arxiv_id) DESC
  LIMIT 20
),

eligible_papers AS (
  SELECT DISTINCT pc.arxiv_id
  FROM arxiv_db.arxiv_paper_categories pc
  JOIN top_categories tc
    ON tc.category = pc.category
  WHERE pc.published_at >= DATE '2020-01-01'
),

affiliations AS (
  SELECT
    regexp_replace(
      lower(trim(affiliation_rows.affiliation.affiliation)),
      '\\s+',
      ' ',
      'g'
    ) AS normalized_affiliation
  FROM eligible_papers ep
  JOIN comet_papers cp
    ON cp.arxiv_id = ep.arxiv_id,
    unnest(from_json(cp.prediction_json, '[
      {
        "name": "VARCHAR",
        "affiliations": [
          {
            "affiliation": "VARCHAR",
            "ror_id": "VARCHAR"
          }
        ]
      }
    ]')) AS author_rows(author),
    unnest(author_rows.author.affiliations) AS affiliation_rows(affiliation)
  WHERE affiliation_rows.affiliation.affiliation IS NOT NULL
    AND trim(affiliation_rows.affiliation.affiliation) <> ''
)

SELECT
  COUNT(*) AS affiliation_mentions,
  COUNT(DISTINCT normalized_affiliation) AS unique_affiliations
FROM affiliations;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [25]:
top20_total_unique_affiliations

,affiliation_mentions,unique_affiliations
0,2383727,494689


Deduplicated across all 20 top categories from 2020 onwards, our DB has $494689$ UNIQUE affilitations.

In [29]:
# getting an ROR

comet_rows[1]['prediction']

[{'name': 'L. Viennot',
  'affiliations': [{'affiliation': 'Laboratoire de didactique André Revuz, PRES Sorbonne Paris Cité, Université Paris Diderot-Paris 7',
    'ror_id': 'https://ror.org/053p8kg34'}]},
 {'name': 'C. de Hosson',
  'affiliations': [{'affiliation': 'Laboratoire de didactique André Revuz, PRES Sorbonne Paris Cité, Université Paris Diderot-Paris 7',
    'ror_id': 'https://ror.org/053p8kg34'}]}]

In [1]:
from arxiv_map.warehouse import RorWarehouse


In [2]:
ror = RorWarehouse()
ror.build()

ror.find_by_ids(["ror:04ttjf776", "https://ror.org/04ttjf776"])

ror.search_names("RMIT University")

,ror_id,ror_url,display_name,name,name_types,lang,country_code,country,region,city,lat,lon
0,04ttjf776,https://ror.org/04ttjf776,RMIT University,RMIT University,"[""ror_display"",""label""]",en,AU,Australia,Victoria,Melbourne,-37.814,144.96332


In [4]:
ror_geo_summary = ror.query("""
SELECT
  COUNT(DISTINCT ror_id) AS total_ror_ids,
  COUNT(DISTINCT ror_id) FILTER (
    WHERE lat IS NOT NULL AND lon IS NOT NULL
  ) AS ror_ids_with_geo,
  COUNT(DISTINCT ror_id) FILTER (
    WHERE lat IS NULL OR lon IS NULL
  ) AS ror_ids_without_geo
FROM ror_institutions
""")

ror_geo_summary

,total_ror_ids,ror_ids_with_geo,ror_ids_without_geo
0,125848,125848,0
